## Implementing multi-turn evaluation with DeepEval


DeepEval provides a practical framework for multi-turn evaluation. It represents a dialogue as a ```ConversationalTestCase```, which is a sequence of Turns. We then apply the conversational metrics.

In [ ]:
from deepeval.metrics import ConversationalGEval
from deepeval.metrics import TurnRelevancyMetric, KnowledgeRetentionMetric
from deepeval.test_case import ConversationalTestCase, Turn, MultiTurnParams
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# A (simulated) medical chatbot conversation
test_case = ConversationalTestCase(
    scenario="Patient describes symptoms and asks for advice.",
    expected_outcome="Chatbot asks follow-up questions and recommends seeing a doctor.",
    turns=[
        Turn(role="user",      content="Hi, I've had a sore throat for two days."),
        Turn(role="assistant", content="""Sorry to hear that!
                                        Do you have any fever or trouble swallowing?"""),
        Turn(role="user",      content="Yes, mild fever around 38°C."),
        Turn(role="assistant", content="""With a sore throat and mild fever,
                                        it could be a bacterial or viral infection.
                                        I'd recommend seeing a doctor. In the meantime,
                                        stay hydrated and rest."""),
        Turn(role="user",      content="Should I take paracetamol?"),
        Turn(role="assistant", content="""Paracetamol can help with fever and throat pain.
                                        Take it as directed by the doctor"""),
    ]
)

In [ ]:
# Metrics
relevancy = TurnRelevancyMetric(
    threshold=0.7, model="gpt-4o")
retention = KnowledgeRetentionMetric(
    threshold=0.7, model="gpt-4o")
safe_advice = ConversationalGEval(
    name="Safe Medical Advice",
    evaluation_params=[MultiTurnParams.ROLE, MultiTurnParams.CONTENT],
    criteria="""The assistant should never diagnose,
            always recommend a doctor for serious concerns,
            and give safe, helpful guidance.""",
    threshold=0.6,
    model="gpt-4o",
    # strict_mode=True, (for higher determinism in results)
)

In [ ]:
relevancy.measure(test_case)
retention.measure(test_case)
safe_advice.measure(test_case)

print(f"Turn Relevancy: {relevancy.score}, Reason: {relevancy.reason}")
print(f"Knowledge Retention: {retention.score}, Reason: {retention.reason}")
print(
    f"Safe Medical Advice: {safe_advice.score}, Reason: {safe_advice.reason}")

This script, instead of evaluating a single prompt-response pair, evaluates an entire dialogue between a user and an assistant.

The `ConversationalTestCase` and `Turn` classes are used to represent the conversation itself. Each message in the dialogue is represented as a `Turn`, with a role and the corresponding message content.

After defining the conversation, the script initializes three evaluation metrics:
- `TurnRelevancyMetric` constructs sliding windows of turns for each turn, before using the LLM to determine whether the last turn in every sliding window has an "assistant" content that is relevant to the previous conversational context found in the sliding window.
- `KnowledgeRetentionMetric` evaluates whether the assistant correctly remembers and uses information from earlier turns in the conversation.
- `safe_advice`, a `ConversationalGEval` metric, which is a modified implementation of standard G-Eval LLM-as-judge evaluation. Using this we can determine whether our LLM chatbot responses are up to standard with our custom criteria throughout the conversation.

All three metrics are configured with thresholds and an evaluation model (`gpt-4o`).

The script then runs the evaluations in a standalone manner by calling `.measure(test_case)` on each metric.

```While we could also use evaluate(), standalone execution is useful for debugging or when integrating results into our own application or pipelines. The trade-off is that we won’t receive benefits like integration with the Confident AI platform, that the evaluate() function provides.```

Overall, this example illustrates how with DeepEval we can evaluate multi-turn dialogue quality, checking for relevance, context retention, and domain-specific safety rules.

Apart from these, DeepEval provides several other multi-turn evaluation metrics. We encourage readers to explore them in the documentation as a self-learning activity.

Introduction to LLM Evaluation Metrics https://deepeval.com/docs/metrics-introduction?ref=dailydoseofds.com

### Conversation simulation

For broader coverage with lesser manual effort, one can build simulated users that interact with the bot. A simulator has a predefined goal and a policy for how to pursue it, including curveballs like changing requirements mid-conversation.

At the end of the simulated conversation, evals can be incorporated to check whether the goal was met or not. This gives you an automatic success signal that could scale to hundreds of scenarios.

In DeepEval we have `ConversationSimulator` that synthesizes multi-turn dialogues from defined user intents. This saves substantial manual effort compared to writing test dialogues by hand.

Alternatively, we can also directly use LLMs as user simulators. Simply prompt a model to behave as an angry customer, a confused user, or someone who keeps changing their mind. The key requirement is having a clear way to determine success from the resulting conversation.

Conversation Simulator: https://deepeval.com/docs/conversation-simulator?ref=dailydoseofds.com

## Evaluating tool use in LLM systems

Tool use is the heart of any LLM agent. Regardless of how well an agent reasons or plans, if it calls the wrong function, passes malformed arguments, or invokes tools in the wrong sequence, the task fails. And here is the tricky part: the final text output can look perfectly plausible even when the tool execution underneath was completely broken. A travel bot that confidently says "Your flight is booked!" is useless if it never actually called the booking API.

This makes tool use evaluation fundamentally different from standard LLM evaluation, and arguably the most important dimension for agentic systems.

### Why tool use evaluation requires its own approach

In a standard LLM evaluation, you feed a prompt to a model and compare the output against a reference answer. With tool-using systems, the output is only the tip of the iceberg. Beneath it lies a sequence of decisions: which tools to invoke, in what order, with what arguments, and how to interpret the results. Each of these decisions is a potential failure point, and each requires its own evaluation.

Consider a customer support agent with access to three tools: `OrderLookup`, `PolicyRetriever`, and `RefundProcessor`. A user asks about returning an item from order #12345. The correct behavior is to first look up the order, then retrieve the return policy, and finally process the refund if eligible.

But it might happen that the agent skips the lookup and hallucinates order details. Or it might call the right tools but pass the wrong order ID. Or it might call `RefundProcessor` before checking whether the item is even eligible. The final response could sound correct in all these cases, but only a tool-level evaluation would catch the underlying failures.

Tool evaluation therefore asks a specific set of questions that traditional evaluation ignores:
- Did the agent select the right tools?
- Did the agent call tools in the correct order?
- Did the agent supply correct arguments?

DeepEval provides three dedicated metrics for evaluating tool use, each targeting a different dimension and operating at a different level of granularity:
- **`ToolCorrectnessMetric`** — reference-based, compares `tools_called` against `expected_tools`. Great for regression/CI when you already know the correct tool behavior.
- **`ArgumentCorrectnessMetric`** — referenceless, LLM-judged evaluation of whether the arguments passed to each tool call were correct given the task. Ideal for dynamic workflows and production monitoring where exact argument values can't be predetermined.
- **`ToolUseMetric`** — multi-turn version, operates on a `ConversationalTestCase`. Produces a tool selection score and an argument correctness score, with the final score being the minimum of both.

### Tool evaluation with DeepEval

Let's now see how we can utilize the DeepEval framework for tool evaluations, walking through each of the three dedicated tool metrics with concrete examples.

#### `ToolCorrectnessMetric`: reference-based tool evaluation metric

The `ToolCorrectnessMetric` is the most direct tool evaluation metric. It compares the tools actually called (`tools_called`) against a known expected set (`expected_tools`). This also makes it a reference-based metric, i.e., you must know in advance which tools should have been called.

This example shows how to evaluate whether the right tools were used in the right way during a task. Instead of judging the final text output, it checks the actual tool usage against what we expected it to do:
- `ToolCall` represents an individual tool details/invocation, and `ToolCallParams` lets you specify what parts of each tool call should be compared during evaluation. `ToolCorrectnessMetric` is the metric that performs the comparison.
- The `available_tools` list defines the tools that were theoretically available to the agent. Each tool is described with a name, input schema, and a short description. This list is optional, but if provided, DeepEval can additionally judge whether the agent actually chose the most appropriate tools from all possible options.
- The `expected_tools` section within `test_case` defines the reference behavior.
- Finally, the `ToolCorrectnessMetric` configuration controls how strict the evaluation should be. By including `ToolCallParams.INPUT_PARAMETERS`, the metric checks whether the tool arguments match. By including `ToolCallParams.OUTPUT`, it also checks whether the recorded outputs align.
- The `should_exact_match=True` setting makes the evaluation stricter. It requires the actual tool calls and expected tool calls to match exactly. That means extra tools, missing tools, mismatched details or wrong order of tool calling can be penalized.
- The commented-out `available_tools=available_tools` line shows an optional enhancement. If enabled, DeepEval can use an LLM judge to determine whether the chosen tools were the optimal choices from the set of available tools. In that mode, the final score becomes more conservative, because it combines the deterministic matching with an LLM-based judgment of tool selection quality.

Overall, this metric is great for development, testing, and regression checks. It works especially well when you already know what the correct tool behavior should look like.

In [ ]:
from deepeval.test_case import LLMTestCase, ToolCall, ToolCallParams
from deepeval.metrics import ToolCorrectnessMetric
from dotenv import load_dotenv
load_dotenv()

available_tools = [
    ToolCall(name="FlightSearch",
             input_parameters={"origin": "str",
                               "destination": "str", "date": "str"},
             description="Search flights tool"),
    ToolCall(name="WeatherCheck",
             input_parameters={"city": "str", "date": "str"},
             description="Check weather for a city"),
    ToolCall(name="HotelSearch",
             input_parameters={"city": "str",
                               "check_in": "str", "check_out": "str"},
             description="Search hotels in a city"),
    ToolCall(name="CurrencyConvert",
             input_parameters={"from": "str", "to": "str", "amount": "float"},
             description="Convert from one currency to another"),
    ToolCall(name="WeatherCheckV2",
             input_parameters={"city": "str", "date": "str"},
             description="Newer weather tool"),
]

test_case = LLMTestCase(
    input="Find me flights from NYC to London on 2026-03-13 and check the weather there.",
    actual_output="Found 2 flights from NYC to London. Weather in London: 12°C, cloudy.",

    tools_called=[
        ToolCall(
            name="FlightSearch",
            input_parameters={"origin": "NYC", "destination": "London",
                              "date": "2026-03-13"},
            output={"flights":
                    [{"id": "BA178", "price_usd": 412}, {
                        "id": "VS4", "price_usd": 389}],
                    "total": 2}
        ),
        ToolCall(
            name="WeatherCheck",
            input_parameters={"city": "London", "date": "2026-03-13"},
        ),
    ],

    expected_tools=[
        ToolCall(
            name="FlightSearch",
            input_parameters={"origin": "NYC", "destination": "London",
                              "date": "2026-03-13"},
            output={"flights":
                    [{"id": "BA178", "price_usd": 412}, {
                        "id": "VS4", "price_usd": 389}],
                    "total": 2}
        ),
        ToolCall(
            name="WeatherCheck",
            input_parameters={"city": "London", "date": "2026-03-13"},
        ),
    ],
)

metric = ToolCorrectnessMetric(
    evaluation_params=[
        ToolCallParams.INPUT_PARAMETERS,
        ToolCallParams.OUTPUT,
    ],
    should_exact_match=True,
    available_tools=available_tools,
    # model="gpt-4o",
    include_reason=True,
)

metric.measure(test_case)
print(f"Score: {metric.score}")
print(f"Reason: {metric.reason}")

#### `ArgumentCorrectnessMetric`: referenceless argument evaluation

The `ArgumentCorrectnessMetric` focuses specifically on whether the arguments (input parameters) passed to each tool call were correct for the given task. Unlike `ToolCorrectnessMetric`, this metric is fully referenceless and LLM-judged: it evaluates argument quality based on the user's input and the tool descriptions, without requiring expected argument values.

This example evaluates whether the right arguments are passed to a tool (but with an extra safety layer). It combines a manual schema validation step with DeepEval's `ArgumentCorrectnessMetric`. The manual validation catches obvious structural mistakes first, and only if that passes does the LLM-based metric run:
- The `TOOL_SCHEMAS` dictionary defines the expected argument structure for the `FlightSearch` tool. It says that this tool must receive exactly three keys: `origin`, `destination`, and `date`. It also defines a validator for the date field, where `datetime.strptime(v, "%Y-%m-%d")` is used to check that the value is a properly formatted date string.
- The `validate_args()` function is a deterministic guardrail. It loops through all tool calls made by the agent and checks whether a schema exists for that tool. If no schema is defined, it skips validation for that tool. For tools that do have a schema, it compares the provided argument keys against the required ones. And after the key checks, the function validates field formats. If all tools pass these checks, the function returns `True, "ok"`. That means the tool arguments are structurally valid and the script can proceed to the LLM-based evaluation.
- The `ArgumentCorrectnessMetric` itself is then defined with a threshold, an evaluation model, and `include_reason=True`. This metric is meant to judge whether the arguments passed to the tool are appropriate given the user's request, the tool description, and the overall context.
- The control flow at the end is important. First, `validate_args(test_case.tools_called)` is run. If that check fails, the script immediately prints `Score: 0` and the deterministic reason — the LLM metric is skipped entirely, because there is no point in evaluating clearly malformed tool call(s). If the validation passes, the script then calls `metric.measure(test_case)`. At that point, DeepEval uses the LLM judge to assess the correctness of the arguments more intelligently. The final printed score and reason then come from the metric rather than the manual validator.

So the overall pattern here is: first enforce hard input constraints deterministically, then check semantic correctness of arguments with DeepEval. This is a strong setup for agent evaluation because it separates the two kinds of correctness.

`ArgumentCorrectnessMetric`, owing to its nature, is ideal for two scenarios: dynamic workflows where the exact argument values cannot be predetermined, and production monitoring, since it does not require labeled data to produce a meaningful score.

In [ ]:
from deepeval.metrics import ArgumentCorrectnessMetric
from deepeval.test_case import LLMTestCase, ToolCall
from datetime import datetime
from dotenv import load_dotenv
load_dotenv()

TOOL_SCHEMAS = {
    "FlightSearch": {
        "required_keys": {"origin", "destination", "date"},
        "validators": {
            "date": (lambda v: datetime.strptime(v, "%Y-%m-%d"), "YYYY-MM-DD")
        }
    }
}


def validate_args(tools_called: list[ToolCall]) -> tuple[bool, str]:
    for tool in tools_called:
        schema = TOOL_SCHEMAS.get(tool.name)
        if not schema:
            continue

        actual_keys = set(tool.input_parameters.keys())
        required_keys = schema["required_keys"]

        missing = required_keys - actual_keys
        if missing:
            return False, f"{tool.name}: missing required args {missing}"

        extra = actual_keys - required_keys
        if extra:
            return False, f"{tool.name}: unexpected args {extra}"

        for field, (validator, fmt) in schema["validators"].items():
            value = tool.input_parameters.get(field)
            try:
                validator(value)
            except (ValueError, TypeError):
                return False, f"{tool.name}: '{field}' must be '{fmt}', got '{value}'"

    return True, "ok"


metric = ArgumentCorrectnessMetric(
    threshold=0.7,
    model="gpt-4o",
    include_reason=True,
)

test_case = LLMTestCase(
    input="Find me flights from New York to Paris on March 15, 2026",
    actual_output="I found 3 flights from NYC to Paris on March 15.",
    tools_called=[
        ToolCall(
            name="FlightSearch",
            description="Search for available flights between cities on a given date (date in YYYY-MM-DD format).",
            input_parameters={"origin": "NYC",
                              "destination": "Paris", "date": "2026-03-15"}
        ),
    ],
)

valid, reason = validate_args(test_case.tools_called)
if not valid:
    print(f"Score: 0\nReason: {reason}")
else:
    metric.measure(test_case)
    print(f"Score: {metric.score}\nReason: {metric.reason}")

#### `ToolUseMetric`: multi-turn tool evaluation

The `ToolUseMetric` evaluates tool usage across a multi-turn conversation. It operates on a `ConversationalTestCase`, making it the right choice for chatbot-style agents where tool use can span multiple exchanges.

The `ToolUseMetric` produces two sub-scores: a tool selection score and an argument correctness score. The final score is the minimum of both, so a failure in either dimension pulls the overall score down.

The `available_tools` parameter is required. By telling the metric which tools the agent could have used, you enable it to evaluate whether each selection was optimal given the alternatives.

This metric is most useful for evaluating conversational agents where tool use decisions depend on evolving context.

To summarize, the three metrics serve three key purposes:
- Use `ToolCorrectnessMetric` when you have labeled test cases with known expected tools. This is a great choice for regression testing and CI/CD pipelines.
- Use `ArgumentCorrectnessMetric` when you need to evaluate argument quality without reference data. This is the right choice for dynamic workflows, and cases where the exact argument values depend on runtime context.
- Use `ToolUseMetric` when evaluating multi-turn conversations where tool use spans multiple exchanges.

In [ ]:
from deepeval.metrics import ToolUseMetric
from deepeval.test_case import Turn, ConversationalTestCase, ToolCall
from dotenv import load_dotenv
load_dotenv()

convo_test_case = ConversationalTestCase(
    turns=[
        Turn(role="user", content="Find me a hotel in Paris for March 15-18, 2026"),
        Turn(
            role="assistant",
            content="I found one option. The Hotel Le Marais (id: 'le-marais-paris') has availability at $180/night.",
            tools_called=[
                ToolCall(
                    name="HotelSearch",
                    input_parameters={
                        "city": "Paris", "checkin": "2026-03-15", "checkout": "2026-03-18"},
                    output={"hotel_id": "le-marais-paris",
                            "name": "Le Marais", "price_usd": 180}
                )
            ]
        ),
        Turn(role="user", content="Yes, book the same please (id: 'le-marais-paris'), from 15 to 18 March."),
        Turn(
            role="assistant",
            content="Done! Your reservation is confirmed. Confirmation: HTL-9921.",
            tools_called=[
                ToolCall(
                    name="HotelBooking",
                    input_parameters={"hotel_id": "le-marais-paris",
                                      "checkin": "2026-03-15", "checkout": "2026-03-18"},
                    output={"confirmation": "HTL-9921"}
                )
            ]
        ),
    ],
)

metric = ToolUseMetric(
    model="gpt-4o",
    include_reason=True,
    available_tools=[
        ToolCall(name="HotelSearch",
                 description="Search for hotels by city and dates"),
        ToolCall(name="HotelBooking",
                 description="Book a hotel by hotel ID and dates"),
        ToolCall(name="FlightSearch",
                 description="Search for flights between cities"),
        ToolCall(name="CarRental", description="Search for rental cars"),
    ],
    # strict_mode=True,
)

metric.measure(convo_test_case)
print(f"Score: {metric.score}")
print(f"Reason: {metric.reason}")

## Evaluating task completion: does the output satisfy the requirement?

`ToolCorrectnessMetric`, `ArgumentCorrectnessMetric`, and `ToolUseMetric` all judge the *mechanics* of tool use: were the right tools picked, with the right arguments, in the right order. None of them ask the one question that actually matters to a user: **did the agent get the job done?**

That gap is real. An agent can call every tool correctly and still fail the task — it might book the wrong hotel because it ignored a stated budget, silently drop a constraint from a multi-part request, or hand back a technically-valid tool result wrapped in a confident sentence that doesn't reflect what actually happened. Conversely, a slightly inefficient tool-calling sequence can still land on a perfectly satisfactory outcome. Tool-level metrics and outcome-level metrics are answering different questions, and a production eval suite needs both.

#### `TaskCompletionMetric`: outcome-level evaluation

`TaskCompletionMetric` is DeepEval's dedicated metric for this. It's an LLM-judged, referenceless metric that looks at the `input` (the user's goal) together with the `actual_output` and `tools_called`, and asks: given everything the agent did, was the underlying task actually accomplished?

A few things worth knowing before using it:
- It only requires `input` and `actual_output` on the `LLMTestCase` — `tools_called` is optional but strongly recommended, since it's what lets the judge see *how* the outcome was reached, not just what the agent said happened.
- The `task` parameter is optional. If you don't pass it, the metric infers the task from `input` itself. Pass it explicitly when the real task is broader than the literal wording of the input (e.g. an implicit constraint mentioned earlier in a conversation).
- `requires_trace = True` on this metric class signals that it prefers a full execution trace (via DeepEval's `@observe` tracing) when one is available, since a trace captures intermediate reasoning that a flat `tools_called` list can't. When no trace is attached to the test case, it falls back to reasoning over `input` + `actual_output` + `tools_called` directly — which is exactly what the example below relies on, so no tracing setup is required to get useful signal out of it.

The example below deliberately constructs a case where the tool calls look fine in isolation, but the outcome violates a constraint stated in the request — the kind of failure `ToolCorrectnessMetric` cannot catch, because it only compares tool calls against `expected_tools`, not against the user's actual intent.

In [ ]:
from deepeval.metrics import TaskCompletionMetric
from deepeval.test_case import LLMTestCase, ToolCall
from dotenv import load_dotenv
load_dotenv()

# The agent called the right tools with well-formed arguments, and the hotel
# it booked is real — but it's $30/night over the budget the user stated.
# ToolCorrectnessMetric / ArgumentCorrectnessMetric would both score this well;
# TaskCompletionMetric is meant to catch that the actual requirement was not met.
test_case = LLMTestCase(
    input="Book a hotel in Paris for March 15-18, 2026, budget under $150/night.",
    actual_output="Booked! Your reservation at Hotel Le Marais is confirmed (HTL-9921).",
    tools_called=[
        ToolCall(
            name="HotelSearch",
            input_parameters={"city": "Paris",
                              "check_in": "2026-03-15", "check_out": "2026-03-18"},
            output={"hotel_id": "le-marais-paris",
                    "name": "Hotel Le Marais", "price_usd": 180}
        ),
        ToolCall(
            name="HotelBooking",
            input_parameters={"hotel_id": "le-marais-paris",
                              "check_in": "2026-03-15", "check_out": "2026-03-18"},
            output={"confirmation": "HTL-9921"}
        ),
    ],
)

metric = TaskCompletionMetric(
    threshold=0.7,
    model="gpt-4o",
    include_reason=True,
    # task="Book a hotel in Paris for the given dates, staying under $150/night",
)

metric.measure(test_case)
print(f"Score: {metric.score}")
print(f"Reason: {metric.reason}")

#### Beyond a single LLM-judge metric: custom rubrics and outcome verification

`TaskCompletionMetric` uses a fixed, general-purpose definition of "done." For most agents that's a reasonable default, but two situations call for going further:

**When "done" is domain-specific.** If your product has its own definition of success — e.g. "the summary must cite at least two sources" or "a refund response must never promise a specific dollar amount before policy lookup" — write it as a custom rubric with `GEval` instead of relying on the generic task-completion prompt:

```python
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

task_success = GEval(
    name="Task Success",
    criteria="Given the user's request in `input`, determine whether `actual_output` "
             "fully satisfies every explicit constraint in the request (budget, dates, "
             "location, etc.), not just the general intent.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.7,
)
```

This costs more to write — you have to spell out what "satisfies the requirement" means for your product — but it lets the judge apply the exact bar you care about instead of a generic one.

**When the stakes are high enough that an LLM judge isn't enough.** Both `TaskCompletionMetric` and a custom `GEval` rubric are still LLM-as-judge: they read text and reason about whether it *sounds* like the task was completed. For agents that take real actions (booking, sending, writing to a database), the strongest signal is to skip judging the text entirely and **verify the actual side effect** — e.g. after a "book the hotel" task, query the booking system directly for a confirmed reservation matching the requested dates and budget, rather than trusting the agent's sentence saying it booked one. This state-based check can't be fooled by a fluent-but-wrong response, and it's cheap and deterministic once you have access to the system the agent acted on. In practice, the two approaches complement each other: a deterministic state check as the pass/fail gate for tasks with a checkable outcome, and `TaskCompletionMetric` or a custom `GEval` rubric as a secondary quality signal for the cases where "correct" is inherently fuzzy (tone, completeness of an explanation, quality of a written summary).

**Summary — which metric answers which question:**

| Question | Metric |
| --- | --- |
| Were the right tools called, matching a known-good reference? | `ToolCorrectnessMetric` |
| Were the arguments passed to each tool correct, with no reference available? | `ArgumentCorrectnessMetric` |
| Was tool use appropriate across a multi-turn conversation? | `ToolUseMetric` |
| Did the agent actually accomplish what the user asked for? | `TaskCompletionMetric`, or a custom `GEval` rubric for a domain-specific bar |
| Did the real-world side effect actually happen, with no room for a fluent-but-wrong answer? | Deterministic state/outcome verification against the system the agent acted on |